# News Articles Summarisation Model
This abstractive summarization model is fine-tuned from `facebook/bart-large-cnn`. The model generates human-like summaries by understanding and paraphrasing news articles rather than extracting sentences.

This model was built using VScode IDE with a Jupyter Notebook extension.

## Pre-requisites
If you are using google colab or a notebook, please uncomment this and install the packages.

In [ ]:
# In Colab or a fresh environment, install dependencies first. Recommended:
# !pip install -q -r requirements.txt
# Or install core packages only, for example:
# !pip install transformers datasets evaluate nltk scikit-learn bert-score accelerate torch

If you are using your local machine, please install the required packages above. Given pip and python are installed; you can run `pip install -r requirements.txt`

## 📁 1. Preparing data

### 1.1 Data cleaning

Mimimal function `pre_process_txt` to remove only unwanted characters and formatting because abstract summarisation requires the original text to remain mostly intact to provides essential context and meaning.

In [ ]:
import re
def pre_process_txt(text):
    """Pre-process text by removing unwanted characters and formatting."""
    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", "", text)

    # Remove non-text characters but keep punctuation marks
    text = re.sub(r"[^\w\s\.,!?;:'\"-]", " ", text)

    # Collapse extra spaces
    cleaned_text = re.sub(r"\s+", " ", text).strip()
    
    return cleaned_text

### 1.2 Reduce text length to fit models token input limit

The `max_position_embedding` for `facebook/bart-large-cnn` is `1024` tokens, The function `reduce_by_importance` takes input text and a default maximum token limit to ensure text is within limit while considering sentence value to not carelessly truncate out important information.

`TfidfVectorizer` with `TF-IDF` vectors is used to calculate sentence importance. The original sentence positioning is also considered as it provides contextual value.

In [ ]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import BartTokenizer

nltk.download("punkt", quiet=True)
try:
    nltk.download("punkt_tab", quiet=True)  # NLTK 3.8+ uses punkt_tab for sent_tokenize
except Exception:
    pass

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")

def reduce_by_importance(text, max_tokens):
    """Reduce text to fit within max_tokens by selecting most important sentences using TF-IDF."""
    # split into sentences
    sentences = nltk.sent_tokenize(text)

    # rank with TF-IDF
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(sentences)
    scores = tfidf_matrix.sum(axis=1).A1

    # sort by importance while tracking original placement
    ranked_indices = sorted(range(len(sentences)), key=lambda i: scores[i], reverse=True)

    selected_sent = []
    total_tokens = 0

    # select sentences until max token limit is reached
    for idx in ranked_indices:
        sent = sentences[idx]
        tokens = len(tokenizer.encode(sent, add_special_tokens=False))
        if total_tokens + tokens <= max_tokens:
            selected_sent.append(idx)
            total_tokens += tokens
        else:
            break

    # sort selected indices to maintain original order
    selected_sent.sort()

    # reconstruct text in original order
    selected = [sentences[i] for i in selected_sent]

    # join selected sentences
    return " ".join(selected)

### 1.3 Load dataset
`pre_process_txt` function loads the dataset by reading all article files and their matching summary files from the given directory. Text is cleaned and truncated if necessary before being added as a dictionary entry.

There is also print statements to highlight information regarding the dataset which can help refine the model and inform model parameter.

A try except block is added to handle any errors that may occur when trying to access and read the files.

In [ ]:
from pathlib import Path

def load_dataset(base_path):
    """Load articles and summaries from the specified base path."""
    articles_dir = Path(base_path) / "Articles"
    summary_dir = Path(base_path) / "Summary"

    if not articles_dir.exists() or not summary_dir.exists():
        raise FileNotFoundError("Articles or Summary directory not found.")

    # Get all article files and sort them
    article_files = sorted(articles_dir.glob("*.txt"))

    dataset = []
    
    for article_path in article_files:
            # Get corresponding summary file
            file_id = article_path.stem  # e.g., "001" from "001.txt"
            summary_path = summary_dir / f"{file_id}.txt"

            # Skip if summary doesn't exist
            if not summary_path.exists():
                print(f"Warning: No summary found for {file_id}")
                continue

            # Try and except block to handle potential read errors
            try:
                with open(article_path, 'r', encoding='utf-8') as f:
                    article = f.read()
                    # article = pre_process_txt(article)
                    article_tokens = len(
                        tokenizer.encode(article, add_special_tokens=False)
                    )
                    if article_tokens > 1024:
                        article = reduce_by_importance(article, max_tokens=1024)
                    article_length = len(
                        tokenizer.encode(article, add_special_tokens=False)
                    )

                with open(summary_path, 'r', encoding='utf-8') as f:
                    summary = f.read()
                    # summary = pre_process_txt(summary)
                    summary_length = len(
                        tokenizer.encode(summary, add_special_tokens=False)
                    )

                dataset.append({
                    'id': file_id,
                    'article': article,
                    'summary': summary,
                    'article_length': article_length,
                    'summary_length': summary_length,
                })

            except Exception as e:
                print(f"Error processing {file_id}: {e}")
    
    print(f"Loaded {len(dataset)} article-summary pairs")
    # Knowing the maximum lengths of articles can help inform model choices and parameters
    print(f"Max article length: {max(d['article_length'] for d in dataset)} tokens")
    # Calculate max and min lengths which will be inform parameters to use for model training
    print(f"Max summary length: {max(d['summary_length'] for d in dataset)} tokens")
    print(f"Min summary length: {min(d['summary_length'] for d in dataset)} tokens")
    
    return dataset


This loads our dataset in the variable `data`.

**Please confirm BASE_PATH variable here!** i.e. {BASE_PATH}/Articles or {BASE_PATH}/Summary.

Some IDEs may require a `/` at the beginning as well.

In [ ]:
# Load dataset
BASE_PATH="training_dataset"

data = load_dataset(BASE_PATH)

**Record the minimum and maximum length of summary to inform the model later on.**

In [ ]:
MAX_LEN = 660
MIN_LEN = 40

### 1.4 Organise cleaned data into distinct groups the model will use

Splits dictionary data into groups and convert to be Dataset compatible to use with the model.

* The dataset was shuffled with a fixed random seed (42) to ensure reproducibility while preventing any ordering biases in the data. 
* Training set, `train_dataset`, will be used to teach the model on how to perform the text summarization task, this will be 80% of the dataset.
* Validation set, `val_dataset`, will be used in the training phase to tune the model, this is 10%.
* Finally, the test set, `test_dataset` will be used to test the trained model developed, this is also 10%.

The training set is the highest portion as it needs to be large enough to achieve meaningful results, maximising model performance. 

In [ ]:
from datasets import Dataset, DatasetDict
import random

# Shuffle for randomness
random.seed(42)  # For reproducibility
random.shuffle(data)

# Calculate split indices
total = len(data)
train_end = int(total * 0.8)  # 80% train
test_end = train_end + int(total * 0.1)  # 10% test
# Remaining 10% for validation

# Split the data
train_items = data[:train_end]
val_items = data[train_end:test_end]
test_items = data[test_end:]

# Create datasets
train_dataset = Dataset.from_dict({
    "article": [item['article'] for item in train_items],
    "summary": [item['summary'] for item in train_items],
})

val_dataset = Dataset.from_dict({
    "article": [item['article'] for item in val_items],
    "summary": [item['summary'] for item in val_items],
})

test_dataset = Dataset.from_dict({
    "article": [item['article'] for item in test_items],
    "summary": [item['summary'] for item in test_items],
})

# Combine into a DatasetDict
dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset,
})

# Print dataset sizes
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 💽 2. Train Model

### 2.1 Tokenization of dataset

The model and tokenizer is defined from `facebook/bart-large-cnn` and the function `preprocess_func` tokenise datasets.
* `max_length = MAX_LEN`maximum number of tokens the summary is allowed to have matches the maximum value in the dataset.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments

# Load pre-trained `facebook/bart-large-cnn` model and tokenizer
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def preprocess_func(dataset_dict):
  """Tokenize the inputs and targets from the dataset."""
  inputs = dataset_dict["article"]
  targets = dataset_dict["summary"]
  model_inputs = tokenizer(inputs, max_length=1024, truncation=True, padding="max_length")
# Target tokenization
  with tokenizer.as_target_tokenizer():
    labels = tokenizer(targets, max_length=MAX_LEN, truncation=True, padding="max_length")

  model_inputs["labels"] = labels["input_ids"]
  return model_inputs

# Apply the preprocessing function to the dataset
tokenized_dataset = dataset_dict.map(preprocess_func, batched=True)

### 2.2 Define Trainer Arguments

This defines the trainer arguments and saves the fine-tuned model to directory "./bart-large-cnn-model".

* `learning_rate` can vary between '3e-5' to '5e-5' for BART. I kept the default value of `5e-5`, a sensible starting point that balances convergence speed and stability. When fine-tuning a pre-trained model it should be lowered if signs of overfitting appear.
* `per_device_train_batch_size` and `per_device_eval_batch_size` default is 8 but I have set it to 2, to fit GPU memory constraints of my personal device.
* `num_train_epochs` is how many full passes the trainer makes over the training set (not the number of layers). A value of 5 gives the model more exposure to a small dataset; lower it if validation loss stops improving or starts getting worse.

In [ ]:
training_args = TrainingArguments(
    output_dir="./bart-large-cnn-model",
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=5,
    logging_dir="./logs",
    eval_strategy="epoch",
    save_strategy="no",
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)


### 2.3 Train Model

In [ ]:
# Start training
trainer.train()
# Saves the fine-tuned model
trainer.save_model("./bart-large-cnn-model")

## 📊 3. Evaluation
### 3.1 Generate summaries with the fine-tuned model

Some justification notes on the parameter used to generate the summaries:
* `num_beams = 5` - Uses beam search to explore multiple candidate summaries and select the best one. Although slower, it generally produces more coherent and higher-quality outputs.
* `do_sample = False` - Disables sampling, giving stable summaries with no added randomness—ideal for consistent, factual outputs.
* `length_penalty = 1.0` - Applies no length penalty, allowing the model to generate longer, more detailed summaries. This supports better evaluation scores by retaining more information.
* `max_length`/ `min_length` - Set using values derived from the training dataset. Matching the dataset’s summary lengths helps the model produce outputs that align with the style and structures it learned.

In [ ]:
import torch

generated_summaries = []
reference_summaries = []

model.eval()
with torch.inference_mode():
    for item in dataset_dict["test"]:
        article = item["article"]
        summary = item["summary"]

        inputs = tokenizer(
            article,
            return_tensors="pt",
            max_length=1024,
            truncation=True,
        ).to(model.device)

        summary_ids = model.generate(
            **inputs,
            num_beams=5,
            do_sample=False,
            length_penalty=1.0,
            max_length=MAX_LEN,
            min_length=MIN_LEN,
        )

        gen_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        generated_summaries.append(gen_summary)
        reference_summaries.append(summary)


### 3.2 Evaluation Metrics

#### 3.2.1 ROUGE (Recall-Oriented Understudy for Gisting Evaluation) Score

ROUGE compares overlapping units such as n-grams, word sequences, and word pairs between the generated summary and the reference. It is widely used for summarization evaluation but its recall-focus, it may mean abstractive text summaries are rated lower because of their use of different words, despite it having the same context. As a result, high-quality paraphrased summaries can receive lower ROUGE scores despite capturing the correct content.

* `ROUGE-1` - Measures unigram overlap
* `ROUGE-2` - Measures bigram overlap
* `ROUGE-L` - Measures how well the order of words in generated summary aligns with the reference.
* `ROUGE-Lsum` - Similar to ROUGE-L, but evaluates multiple sentences instead

In [ ]:
from evaluate import load


def evaluate_rouge(ref_list, gen_list):
    rouge = load("rouge")
    results = rouge.compute(predictions=gen_list, references=ref_list)
    return results

#### 3.2.2 BLEU (Bilingual Evaluation Understudy)

BLEU measures how precise the n-grams on the generated text is against the reference text. This means a higher rate is achieved if the generated summary is more similar to the reference summary, linguistically. The text's semantic meaning is not considered.
* `bleu` - overall BLEU score measuring linguistic similarity to reference
* `precisions` - modified precision scores for 1-gram to 4-gram overlaps between the generated text and the reference.
* `brevity_penalty` - penalty applied when generated text is shorter than reference
* `length_ratio` - ratio of the generated text length to reference text length
* `translation_length` - total tokens in generated text
* `reference_length` - total tokens in reference text

In [ ]:
bleu = load('bleu')

def evaluate_bleu(ref_list, gen_list):
    results = bleu.compute(predictions=gen_list, references=ref_list)
    return results

#### 3.2.3 BERTScore
[BERTscore](https://arxiv.org/abs/1904.09675) is an automatic evaluation metric for text generation. Compared to BLEU and ROUGE, BERTScore scores semantic text similarity rather than relying on word matches. The generated text's quality is evaluated, it's semantic focus also makes the judgement aspect more human.
* `Precision` (`P`) - measures the generated summary's relevance to reference
* `Recall` (`R`) - measures how much of the reference content is found in the generated summary
* `F1-score` (`F1`) - Overall score

In [ ]:
from bert_score import BERTScorer

def evaluate_bertscore(ref_list, gen_list):
    scorer = BERTScorer(lang='en')
    P, R, F1 = scorer.score(gen_list, ref_list)  # candidates first
    return P, R, F1

P, R, F1 = evaluate_bertscore(reference_summaries, generated_summaries)

### 3.3 Evaluation Results

In [ ]:
# ROUGE Scores
rouge_scores = evaluate_rouge(reference_summaries, generated_summaries)

print("\n📊 ROUGE SCORES (Content Overlap & Recall-Oriented)")
print(f"  ROUGE-1 (Unigram Match):        {rouge_scores['rouge1']}")
print(f"  ROUGE-2 (Bigram Match):         {rouge_scores['rouge2']}")
print(f"  ROUGE-L (Longest Sequence):     {rouge_scores['rougeL']}")
print(f"  ROUGE-Lsum (Summary-level):     {rouge_scores['rougeLsum']}")

# BLEU Score Section
bleu_scores = evaluate_bleu(reference_summaries, generated_summaries)

print("\n📈 BLEU SCORE (Precision-Oriented Translation Quality)")
print(f"  Overall BLEU Score:             {bleu_scores['bleu']}")
print(f"  Translation Length:             {bleu_scores['translation_length']}")
print(f"  Reference Length:               {bleu_scores['reference_length']}")
print(f"  Length Ratio:                   {bleu_scores['length_ratio']}")
print(f"  Brevity Penalty:                {bleu_scores['brevity_penalty']}")
print("\n  N-gram Precision Breakdown:")
print(f"    • 1-gram (word accuracy):     {bleu_scores['precisions'][0]}")
print(f"    • 2-gram (phrase accuracy):   {bleu_scores['precisions'][1]}")
print(f"    • 3-gram (sequence accuracy): {bleu_scores['precisions'][2]}")
print(f"    • 4-gram (fluency):           {bleu_scores['precisions'][3]}")

# BERTScore Section
precision = P.mean().item()
recall = R.mean().item()
f1 = F1.mean().item()

print("\n🎯 BERTScore")
print(f"  Precision:                      {precision:}")
print(f"  Recall:                         {recall:}")
print(f"  F1 Score:                       {f1:}")

### Read a sample result

In [ ]:
# Display a random example of reference vs generated summary
random_idx = random.randint(0, len(generated_summaries) - 1)
print("Reference Summary:", reference_summaries[random_idx])
print("Generated Summary:", generated_summaries[random_idx])